In [2]:
path="imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"
import pandas as pd
import numpy as np
df = pd.read_csv(path)
print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [3]:
df=df.iloc[:10000]
print(df.shape)
print(df.head())


(10000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [4]:
print(df["sentiment"].value_counts())
print(df.isnull().sum())


sentiment
positive    5028
negative    4972
Name: count, dtype: int64
review       0
sentiment    0
dtype: int64


In [5]:
print(df.duplicated().sum())
print(df.drop_duplicates(inplace=True))
print(df.duplicated().sum())

17
None
0


In [6]:
import re
def remove_html_tags(text): # remove html tags from the text
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)
df['review'] = df['review'].apply(remove_html_tags)
print(df.head())


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. The filming tec...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [7]:
df['review']=df['review'].str.lower()
print(df.head())

                                              review sentiment
0  one of the other reviewers has mentioned that ...  positive
1  a wonderful little production. the filming tec...  positive
2  i thought this was a wonderful way to spend ti...  positive
3  basically there's a family where a little boy ...  negative
4  petter mattei's "love in the time of money" is...  positive


In [8]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
sw_list=stopwords.words('english')
#df['review']=df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (sw_list)]))
def remove_stopwords(x):
    words = x.split()
    
    filtered_words = []
    for word in words:
        if word not in sw_list:
            filtered_words.append(word)
    
    return ' '.join(filtered_words)

df['review'] = df['review'].apply(remove_stopwords)
print(df.head()) 



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91955\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                              review sentiment
0  one reviewers mentioned watching 1 oz episode ...  positive
1  wonderful little production. filming technique...  positive
2  thought wonderful way spend time hot summer we...  positive
3  basically there's family little boy (jake) thi...  negative
4  petter mattei's "love time money" visually stu...  positive


In [9]:
x=df.iloc[:,0:1]
y=df["sentiment"]
print(x.shape)
print(x.head())
print(y.shape)
print(y.head())


(9983, 1)
                                              review
0  one reviewers mentioned watching 1 oz episode ...
1  wonderful little production. filming technique...
2  thought wonderful way spend time hot summer we...
3  basically there's family little boy (jake) thi...
4  petter mattei's "love time money" visually stu...
(9983,)
0    positive
1    positive
2    positive
3    negative
4    positive
Name: sentiment, dtype: object


In [10]:
# converint sentiment to binary values
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)
print(y)

[1 1 1 ... 0 0 1]


In [11]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=1)
print(x_train.shape)
print(x_test.shape)

(7986, 1)
(1997, 1)


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()
x_train=cv.fit_transform(x_train['review']).toarray()
x_test=cv.transform(x_test['review']).toarray()
print(x_train)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [13]:
from sklearn.naive_bayes import GaussianNB
gnb=GaussianNB()
gnb.fit(x_train,y_train)
y_pred=gnb.predict(x_test)
from sklearn.metrics import accuracy_score,confusion_matrix
print(accuracy_score(y_test,y_pred))

0.6324486730095142


In [14]:
from sklearn.naive_bayes import MultinomialNB
mnb=MultinomialNB()
mnb.fit(x_train,y_train)
y_pred=mnb.predict(x_test)
from sklearn.metrics import accuracy_score,confusion_matrix
print(accuracy_score(y_test,y_pred))

0.8487731597396094


In [15]:
# 🔹 1. Imports
import pandas as pd
import numpy as np
import nltk

from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# 🔹 2. Download NLTK data
nltk.download('punkt')


# Convert labels
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Remove rows where mapping failed
df = df.dropna(subset=['sentiment'])

# Lowercase
df['review'] = df['review'].str.lower()


# 🔹 4. Train-test split
x_train, x_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=1
)


# 🔹 5. Sentence + Word Tokenization (YOUR FORMAT)
story = []

for review in x_train:
    sentences = sent_tokenize(review)   # sentence tokenization
    
    for sent in sentences:
        words = word_tokenize(sent)     # word tokenization
        story.append(words)


# 🔹 Check format
print(story[:5])


# 🔹 6. Train Word2Vec
w2v_model = Word2Vec(
    story,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)


# 🔹 7. Convert review → vector
def review_vector(review):
    sentences = sent_tokenize(review)
    vectors = []
    
    for sent in sentences:
        words = word_tokenize(sent)
        
        for word in words:
            if word in w2v_model.wv:
                vectors.append(w2v_model.wv[word])
    
    if len(vectors) == 0:
        return np.zeros(100)
    
    return np.mean(vectors, axis=0)


# 🔹 8. Convert train & test
x_train_vec = np.array([review_vector(text) for text in x_train])
x_test_vec = np.array([review_vector(text) for text in x_test])


# 🔹 9. Train GaussianNB
rf = RandomForestClassifier()
rf.fit(x_train_vec, y_train)


# 🔹 10. Predict
y_pred = rf.predict(x_test_vec)


# 🔹 11. Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\91955\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[['waiting', 'superhero', 'movie', 'like', 'long', 'time', '.'], ['``', 'mystery', 'men', "''", 'takes', 'place', 'among', 'classic', 'comic-strip', 'spoofs', 'tv', 'like', '``', 'batman', "''", '``', 'captain', 'nice', "''", 'cartoons', 'like', '``', 'underdog', "''", '``', 'super', 'chicken', '.', "''"], ['spirit', 'lives', 'them', ':', 'comic', 'tongue-in-cheek', 'tone', ';', 'courage', 'aim', 'heroic', 'life', 'risk', 'looking', 'ridiculous', ';', 'not-so-sure-footed', 'way', 'characters', 'manage', 'prevail', 'adversaries', '.'], ['misfired', 'spark', 'nobility', 'igniting', 'weak', 'ordinary', ',', 'wonderful', 'see', 'glow', 'high', 'bright', 'here', '.'], ['``', 'mystery', 'men', "''", 'opens', 'party', 'nursing', 'home', '.']]
Accuracy: 0.7526289434151227
